# Projeto II — Sprint II | Testes de hipótese

**Tema:** tempo de carregamento de caminhões com grãos.

Faça upload das duas planilhas Excel quando solicitado. A Planilha 1 contém os dados anteriores à intervenção (referência) e a Planilha 2 contém os dados posteriores.

A comparação antes/depois é observacional e, isoladamente, não comprova causalidade.

## 1. Estatísticas descritivas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import files
from io import BytesIO

COLUNA = "Tempo de carregamento (min)"

print("Envie a Planilha 1 (antes).")
u1 = files.upload()
df1 = pd.read_excel(BytesIO(next(iter(u1.values()))))
print("Envie a Planilha 2 (depois).")
u2 = files.upload()
df2 = pd.read_excel(BytesIO(next(iter(u2.values()))))

for nome, df in [("Planilha 1", df1), ("Planilha 2", df2)]:
    if COLUNA not in df.columns:
        raise ValueError(f"{nome}: coluna {COLUNA!r} não encontrada. Colunas disponíveis: {df.columns.tolist()}")

antes = pd.to_numeric(df1[COLUNA], errors="coerce").dropna()
depois = pd.to_numeric(df2[COLUNA], errors="coerce").dropna()
print("n antes:", len(antes), "| n depois:", len(depois))

In [ ]:
resumo = pd.DataFrame({
    "Cenário": ["Antes", "Depois"],
    "n": [len(antes), len(depois)],
    "Média (min)": [antes.mean(), depois.mean()],
    "Desvio-padrão amostral": [antes.std(ddof=1), depois.std(ddof=1)]
})
display(resumo.round(4))

## 2. Teste de hipótese para a média

Teste unilateral à esquerda, pois a hipótese de interesse é redução do tempo médio:

- H₀: μ = μ₀
- H₁: μ < μ₀

μ₀ é a média anterior. Como σ populacional não foi fornecido, o código usa o desvio-padrão amostral anterior como estimativa de referência para reproduzir o teste z da aula. Declare essa suposição na entrega.

In [ ]:
mu0 = antes.mean()
xbarra = depois.mean()
n = len(depois)
sigma_ref = antes.std(ddof=1)

z_calc = (xbarra - mu0) / (sigma_ref / np.sqrt(n))
p_valor = stats.norm.cdf(z_calc)

print(f"μ0 = {mu0:.4f} min")
print(f"x̄ = {xbarra:.4f} min")
print(f"s de referência = {sigma_ref:.4f} min")
print(f"z calculado = {z_calc:.4f}")
print(f"p-valor unilateral à esquerda = {p_valor:.6g}")

## 3. Decisão para α = 5% e α = 1%

In [ ]:
def testar_media(alpha):
    zcrit = stats.norm.ppf(alpha)
    rejeita = z_calc <= zcrit
    print(f"\\nα = {alpha:.0%}")
    print(f"Valor crítico: {zcrit:.4f}")
    print(f"Região crítica: z ≤ {zcrit:.4f}")
    print(f"z calculado: {z_calc:.4f}")
    print("Decisão:", "REJEITAR H0" if rejeita else "NÃO REJEITAR H0")
    print("Conclusão:", ("há evidência de redução da média" if rejeita
                        else "não há evidência suficiente de redução da média"))
    return zcrit

zcrit_5 = testar_media(0.05)
zcrit_1 = testar_media(0.01)

## 4. Gráficos das regiões críticas

In [ ]:
def grafico_media(alpha):
    zcrit = stats.norm.ppf(alpha)
    x = np.linspace(-4, 4, 1000)
    y = stats.norm.pdf(x)
    plt.figure(figsize=(9, 4.5))
    plt.plot(x, y, label="Normal padrão")
    plt.fill_between(x, 0, y, where=x <= zcrit, alpha=.3, label="Região de rejeição")
    plt.axvline(zcrit, ls="--", label=f"z crítico = {zcrit:.3f}")
    plt.axvline(z_calc, ls="-.", label=f"z calculado = {z_calc:.3f}")
    plt.title(f"Teste unilateral à esquerda — α={alpha:.0%}")
    plt.xlabel("z"); plt.ylabel("Densidade"); plt.legend(); plt.grid(alpha=.2); plt.show()

grafico_media(.05)
grafico_media(.01)

## 5. Conclusão e comparação

Compare a decisão inferencial com a redução descritiva da média entre as planilhas. **Não rejeitar H₀ não significa aceitar ou provar H₀.** A análise antes/depois não demonstra causalidade por si só. Registre a suposição sobre σ e confirme com o docente se o teste z é o procedimento exigido quando o desvio-padrão populacional é desconhecido.